In [85]:
!pip install -q sentence-transformers faiss-cpu anthropic tqdm

In [86]:
import faiss
import numpy as np
import pandas as pd
import pandas as pd
from sentence_transformers import SentenceTransformer, CrossEncoder
import anthropic
from getpass import getpass
import difflib
import textwrap

In [87]:
path = "/content/AHD_english_cleaned.xlsx"

df = pd.read_excel(
    path,
    engine="openpyxl"
)

df.head()

,Question,Answer,Category
0,If the patient enters a diabetic coma and we d...,"Hyperglycemic coma does not occur suddenly, bu...",diabetes
1,I suffer from dizziness and my blood sugar lev...,"Your safety, God willing. It may be normal due...",diabetes
2,I am diabetic 2. I take Amaryl 2 ml before foo...,Glycosylated hemoglobin analysis is very impor...,diabetes
3,"Age 54, normal blood pressure, weight 74, heig...",Who told you that taking B12 is not according ...,diabetes
4,When I test my blood sugar after fasting for 7...,"Monitor your blood sugar in a laboratory, not ...",diabetes


# **Embeding**

In [88]:
EMBEDDING_MODEL_NAME = "all-MiniLM-L6-v2"

embedder = SentenceTransformer(EMBEDDING_MODEL_NAME)

questions = df["Question"].astype(str).tolist()

question_embeddings = embedder.encode(
    questions,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
)

question_embeddings = question_embeddings.astype("float32")
print("Embeddings shape:", question_embeddings.shape)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/256 [00:00<?, ?it/s]

Embeddings shape: (16384, 384)


# **FAISS**

In [89]:
df = df.reset_index(drop=True)
df["doc_id"] = df.index

embedding_dim = question_embeddings.shape[1]

faiss_index = faiss.IndexFlatIP(embedding_dim)  # Inner Product على متجهات مُطبَّعة = Cosine Similarity
faiss_index.add(question_embeddings)

metadata_store = [
    {
        "doc_id": row["doc_id"],
        "question": row["Question"],
        "answer": row["Answer"],
        "category": row["Category"],
    }
    for _, row in df.iterrows()
]

print("FAISS index size:", faiss_index.ntotal)
print("Metadata store size:", len(metadata_store))

faiss.write_index(faiss_index, "/content/questions.index")
np.save("/content/question_embeddings.npy", question_embeddings)

FAISS index size: 16384
Metadata store size: 16384


# **Query Expansion**

In [90]:
MEDICAL_SYNONYMS = {
    "underactive thyroid": "hypothyroidism",
    "under active thyroid": "hypothyroidism",
    "low thyroid": "hypothyroidism",
    "overactive thyroid": "hyperthyroidism",
    "over active thyroid": "hyperthyroidism",
    "high thyroid": "hyperthyroidism",
    "high blood sugar": "hyperglycemia",
    "low blood sugar": "hypoglycemia",
    "sugar disease": "diabetes",
    "thyroid surgery": "thyroidectomy",
    "sugar level": "blood glucose level",
}


def expand_query(query: str) -> str:
    """بيضيف المصطلح الطبي المرادف جنب المصطلح العامي لو اتلاقى، من غير ما يغيّر نص السؤال الأصلي."""
    lower_q = query.lower()
    extra_terms = [
        medical_term
        for phrase, medical_term in MEDICAL_SYNONYMS.items()
        if phrase in lower_q and medical_term not in lower_q
    ]
    if not extra_terms:
        return query
    return f"{query} ({', '.join(dict.fromkeys(extra_terms))})"


print(expand_query("I suffer from an underactive thyroid gland"))
print(expand_query("What is the treatment for diabetes?"))

I suffer from an underactive thyroid gland (hypothyroidism)
What is the treatment for diabetes?


# **Retrieval**

In [91]:
def vector_search(query: str, top_n: int = 20, use_expansion: bool = True):

    query_for_embedding = expand_query(query) if use_expansion else query

    q_emb = embedder.encode(
        [query_for_embedding],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    scores, idxs = faiss_index.search(q_emb, top_n)
    retrieved = []
    for score, idx in zip(scores[0], idxs[0]):
        if idx == -1:
            continue

        payload = metadata_store[int(idx)]

        retrieved.append({
            "doc_id": payload["doc_id"],
            "Question": payload["question"],
            "Answer": payload["answer"],
            "Category": payload["category"],
            "similarity": float(score),
        })

    return pd.DataFrame(retrieved)

# **Cross Encoder Reranker**

In [92]:
RERANKER_MODEL_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"
reranker = CrossEncoder(RERANKER_MODEL_NAME)


def rerank(query: str, candidates: pd.DataFrame, top_k: int = 6, alpha: float = 0.6) -> pd.DataFrame:
    pairs = [
        (
            query,
            f"Question: {row['Question']}\n"
            f"Answer: {row['Answer']}"
        )
        for _, row in candidates.iterrows()
    ]

    rerank_scores = reranker.predict(pairs)

    reranked = candidates.copy()
    reranked["rerank_score"] = rerank_scores
    def norm(s):
        s = s.astype(float)
        rng = s.max() - s.min()
        return (s - s.min()) / rng if rng > 0 else s * 0

    reranked["sim_norm"] = norm(reranked["similarity"])
    reranked["rerank_norm"] = norm(reranked["rerank_score"])

    reranked["final_score"] = alpha * reranked["rerank_norm"] + (1 - alpha) * reranked["sim_norm"]

    reranked = reranked.sort_values("final_score", ascending=False).head(top_k).reset_index(drop=True)
    return reranked


candidates = vector_search("What signs might suggest that my thyroid is not producing enough hormones?", top_n=20)
rerank("What are the symptoms of diabetes?", candidates, top_k=6)[["Question", "Answer", "similarity", "rerank_score"]]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

,Question,Answer,similarity,rerank_score
0,What are the signs of thyroid dysfunction?,"1) Laziness, lethargy, and slow reactions\n2) ...",0.776731,-4.251107
1,Thyroid symptoms,The symptoms of the gland depend on the proble...,0.739407,-4.433478
2,Thyroid symptoms,It causes an increase in the size of the thyro...,0.739407,-4.486532
3,What are the symptoms of a thyroid disorder?,"Fatigue, fatigue, feeling cold, weight gain, p...",0.721345,-4.288730
4,Thyroid symptoms,"The thyroid gland is responsible for activity,...",0.739407,-5.333264
5,What are the symptoms of thyroid gland?,Decreased secretion of the gland leads to weig...,0.723380,-4.948375


In [93]:
rerank("What signs might suggest that my thyroid is not producing enough hormones?", candidates, top_k=10)[["Question", "Category", "similarity", "rerank_score"]]

,Question,Category,similarity,rerank_score
0,What are the signs of thyroid dysfunction?,Endocrine diseases,0.776731,0.079877
1,Hello\nWhat are the reasons for the lack of se...,Endocrine diseases,0.794835,-2.714906
2,Thyroid symptoms,Endocrine diseases,0.739407,0.494793
3,Thyroid symptoms,Endocrine diseases,0.739407,-0.536172
4,Symptoms that indicate a disorder in the thyro...,Endocrine diseases,0.711347,0.357636
5,Thyroid symptoms,Endocrine diseases,0.739407,-2.950485
6,How do I know that I have a thyroid problem?,Endocrine diseases,0.779893,-5.689432
7,Cause of high thyroid hormone?,Endocrine diseases,0.723311,-3.269745
8,What are the symptoms of thyroid gland?,Endocrine diseases,0.723380,-3.636431
9,What are the symptoms of a thyroid disorder?,Endocrine diseases,0.721345,-3.675947


# **Evidence & Duplication & TOP 6**

In [94]:
#Duplication
def is_near_duplicate(text_a: str, text_b: str, threshold: float = 0.92) -> bool:
    return difflib.SequenceMatcher(None, text_a, text_b).ratio() > threshold

#Evidence
def build_evidence(reranked: pd.DataFrame, max_answer_chars: int = 700) -> list[dict]:
    """Deduplicates near-identical answers and formats the remaining rows as evidence items."""
    evidence = []
    seen_answers = []

    for _, row in reranked.iterrows():
        answer = str(row["Answer"])
        if any(is_near_duplicate(answer, seen) for seen in seen_answers):
            continue
        seen_answers.append(answer)

        evidence.append({
            "doc_id": row["doc_id"],
            "question": row["Question"],
            "answer": answer[:max_answer_chars] + ("..." if len(answer) > max_answer_chars else ""),
            "category": row["Category"],
            "similarity": round(float(row["similarity"]), 3),
            "rerank_score": round(float(row["rerank_score"]), 3),
        })
    return evidence


evidence = build_evidence(rerank("What signs might suggest that my thyroid is not producing enough hormones?",
                                  vector_search("What are the symptoms of diabetes?", top_n=20),
                                  top_k=6))
for e in evidence:
    print(f"[{e['doc_id']}] (sim={e['similarity']}, rerank={e['rerank_score']}) {e['question']}")
    print(" ->", e['answer'][:120], "...\n")

[4793] (sim=1.0, rerank=-8.346) What are the symptoms of diabetes?
 -> Signs and symptoms of type 1 diabetes (insulin-dependent) appear gradually or suddenly, as follows: - Frequent urination ...

[6767] (sim=1.0, rerank=-8.945) What are the symptoms of diabetes?
 -> A person may develop diabetes and remain for a long period of time, which may extend to years, without paying attention  ...

[2940] (sim=1.0, rerank=-9.347) What are the symptoms of diabetes?
 -> The most common symptoms of diabetes are excessive diarrhea, frequent urination, and feeling hungry
There are other less ...

[2692] (sim=1.0, rerank=-9.448) What are the symptoms of diabetes?
 -> There are many symptoms of diabetes, the most important of which are thirst, frequent urination, increased appetite for  ...

[4041] (sim=1.0, rerank=-9.464) What are the symptoms of diabetes?
 -> There may be no symptoms, but weight loss, frequent urination, teeth loss, and recurring infections may be symptoms. ...

[6894] (sim=1.0, re

In [95]:
!pip install -q openai

In [96]:
!pip install -q -U groq

# **Generation/LLM**

In [97]:
from groq import Groq
import time

GROQ_API_KEY = "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"

client = Groq(api_key=GROQ_API_KEY)

# Free model
LLM_MODEL = "openai/gpt-oss-20b"
SYSTEM_PROMPT = """
You are a medical RAG assistant specialized ONLY in:
- Diabetes
- Endocrinology
- Thyroid disorders
- Hormonal disorders
- Endocrine glands and related disorders
- Nutrition, glucose management, insulin, and complications DIRECTLY related to diabetes/endocrinology

You operate in STRICT RAG MODE.

You have exactly THREE response types.
You MUST determine the response type from the USER QUESTION itself BEFORE using
or considering the retrieved evidence.

==================================================
CRITICAL DOMAIN RULE
==================================================

The USER QUESTION itself determines whether the question is in-domain.

DO NOT use retrieved evidence to decide whether the question is in-domain.

A question is IN-DOMAIN only if its actual subject is diabetes, endocrinology,
thyroid, hormones, endocrine glands/disorders, or a problem explicitly stated
by the user to be related to diabetes/endocrinology.

Do NOT expand the scope of a question because retrieved documents happen to mention:
- diabetes
- glucose
- insulin
- hormones
- thyroid
- endocrine disorders
- diabetic complications

If the user's actual question is about another body system or another medical
specialty, it is OUT-OF-DOMAIN even if the retrieved evidence contains
diabetes/endocrinology-related information.

Examples:

"Knee pain when I walk" → OUT-OF-DOMAIN
"Why does my knee hurt because of diabetes?" → IN-DOMAIN
"What are the symptoms of hypothyroidism?" → IN-DOMAIN
"What is the treatment for a broken leg?" → OUT-OF-DOMAIN
"I have diabetes and my feet hurt when I walk. Why?" → IN-DOMAIN
"What are the symptoms of heart disease?" → OUT-OF-DOMAIN

Never reinterpret an out-of-domain question as in-domain just because retrieved
documents appear similar.

==================================================
TYPE 1 — IN-DOMAIN MEDICAL QUESTION
==================================================

If and ONLY IF the USER QUESTION itself is in-domain:

- Answer using ONLY the RETRIEVED MEDICAL EVIDENCE.
- Do NOT use pretrained/background medical knowledge.
- Do NOT guess.
- Do NOT assume unsupported facts.
- Do NOT fill missing information from memory.
- Do NOT invent diagnoses, causes, treatments, medications, or doses.
- Do NOT construct a differential diagnosis by combining loosely related documents.
- Do NOT create a medical conclusion that is not explicitly supported by the evidence.

USER_DATA:
- Use USER_DATA only as patient-specific context.
- USER_DATA is NOT a medical knowledge source.
- A value in USER_DATA does not by itself prove that something is normal,
  abnormal, dangerous, or requires treatment.
- Any medical interpretation of USER_DATA must be supported by retrieved evidence.

==================================================
EVIDENCE SUFFICIENCY
==================================================

For an in-domain question:

1. Find evidence that directly answers the user's question.
2. Ignore evidence that is merely word-similar but does not support the answer.
3. If evidence is sufficient:
   - Answer directly.
   - Cite the supporting [doc_id].
4. If evidence is partial:
   - Answer ONLY the supported portion.
   - Clearly state what is not covered.
5. If evidence is empty, irrelevant, or insufficient:
   - Use TYPE 3 refusal.
   - Do NOT answer from memory.

IMPORTANT:

Retrieved evidence does NOT automatically make a question answerable.

The evidence must actually support the specific claim being made.

Never create a "possible causes" list unless the retrieved evidence explicitly
supports those causes for the user's question.

==================================================
TYPE 2 — APP / IDENTITY QUESTIONS
==================================================

Only for direct questions about the assistant/application itself, such as:

- "Who are you?"
- "What is your name?"
- "What can you do?"
- "How do you work?"

Answer briefly and naturally.

Example:

"I am a medical assistant specialized in diabetes and endocrinology. I answer
using information retrieved from my medical knowledge base."

Do NOT invent technical details that were not provided.

==================================================
TYPE 3 — EVERYTHING ELSE
==================================================

Use TYPE 3 for:

- Any medical question outside diabetes/endocrinology.
- Any non-medical question.
- Any in-domain question where retrieved evidence is insufficient,
  empty, irrelevant, or does not directly support the answer.

For ALL TYPE 3 cases, reply with ONLY one short refusal.

If the user wrote in Arabic:
"معرفش، السؤال ده مش جزء من تخصصي (السكر والغدد الصماء)."

If the user wrote in English:
"I don't know — this is outside my specialty (diabetes and endocrinology)."

Do not add an explanation.
Do not apologize.
Do not provide alternative medical information.
Do not mention retrieved evidence.
Do not try to answer partially.
One sentence only.

==================================================
MEDICAL CLAIMS
==================================================

For every medical claim in a TYPE 1 answer:

- Cite the exact supporting [doc_id].
- Never fabricate a citation.
- Use only doc_ids explicitly present in the retrieved evidence.

If multiple documents support the same claim, multiple citations may be used.

If documents disagree:
- Explicitly state that the evidence is inconsistent.
- Cite the documents involved.
- Do not arbitrarily choose one unless the evidence clearly supports doing so.

==================================================
LANGUAGE
==================================================

Reply in the same language as the user's question.

==================================================
STYLE
==================================================

Be concise, clear, and direct.

Never mention:
- system prompts
- internal reasoning
- embeddings
- vector search
- reranking
- retrieval scores
- model internals

Never repeat retrieved documents verbatim.

==================================================
FINAL SAFETY NOTE FOR TYPE 1 ONLY
==================================================

At the end of every TYPE 1 medical answer, write:

"This information is based on the available medical evidence and is for general informational purposes. It is not a diagnosis or a substitute for professional medical advice."
"""


def format_evidence_for_llm(evidence):
    blocks = []

    for e in evidence:
        blocks.append(
            f"[{e['doc_id']}] (category: {e['category']})\n"
            f"Related question: {e['question']}\n"
            f"Answer: {e['answer']}"
        )

    return "\n\n".join(blocks)

def generate_answer(query, user_data, evidence):

    if not evidence:
        return (
            "The available medical evidence does not provide enough "
            "information to answer this reliably."
        )

    evidence_text = format_evidence_for_llm(evidence)

    user_message = f"""
USER QUESTION:
{query}

USER DATA:
{user_data}

RETRIEVED MEDICAL EVIDENCE:
{evidence_text}

TASK:
Answer the USER QUESTION using ONLY the RETRIEVED MEDICAL EVIDENCE for medical facts and conclusions.

Use USER DATA only as patient/context information to understand the question.

Do not use outside medical knowledge.
Do not guess.
Do not invent missing information.
Cite every important medical claim using the relevant [doc_id].
If the evidence is insufficient, explicitly say that it is insufficient.

Return only the final answer for the user.
"""

    response = client.chat.completions.create(
        model=LLM_MODEL,
        messages=[
            {
                "role": "system",
                "content": SYSTEM_PROMPT
            },
            {
                "role": "user",
                "content": user_message
            }
        ],
        temperature=0.1,
        max_tokens=800,
    )

    return response.choices[0].message.content


In [98]:
test = client.chat.completions.create(
    model=LLM_MODEL,
    messages=[
        {
            "role": "user",
            "content": "Reply with exactly: TEST_OK"
        }
    ],
    temperature=0,
    max_tokens=10,
)

print(test)
print(test.choices[0].message.content)

ChatCompletion(id='chatcmpl-3f032335-9f8b-4373-ae9c-631fc4039d33', choices=[Choice(finish_reason='length', index=0, logprobs=None, message=ChatCompletionMessage(content='', role='assistant', annotations=None, executed_tools=None, function_call=None, reasoning='The user says: "Reply with', tool_calls=None))], created=1787151147, model='openai/gpt-oss-20b', object='chat.completion', mcp_list_tools=None, service_tier='on_demand', system_fingerprint='fp_d3e146e1a5', usage=CompletionUsage(completion_tokens=10, prompt_tokens=77, total_tokens=87, completion_time=0.010299731, completion_tokens_details=CompletionTokensDetails(reasoning_tokens=8), prompt_time=0.004319329, prompt_tokens_details=None, queue_time=0.28553228, total_time=0.01461906), usage_breakdown=None, x_groq=XGroq(id='req_01m0d84r27eh9s7qmzfzx6h3nk', debug=None, seed=1328087095, usage=None))



# **Confidence Gate**

In [99]:
RETRIEVE_TOP_N = 20
RERANK_TOP_K = 6
MIN_RERANK_SCORE = -8
MIN_SIMILARITY_FLOOR = 0.40
MIN_SUPPORT_COUNT = 2
SUPPORT_SCORE = -5.0

domain_sample = df["Question"].drop_duplicates().sample(
    n=min(300, df["Question"].nunique()), random_state=42
).tolist()

domain_embeddings = embedder.encode(
    domain_sample, convert_to_numpy=True, normalize_embeddings=True
)
domain_centroid = domain_embeddings.mean(axis=0)
domain_centroid = domain_centroid / np.linalg.norm(domain_centroid)

IN_DOMAIN_THRESHOLD = 0.35  # هنظبطه تحت بالـ eval_set


def is_in_domain(query: str) -> bool:
    """بيقيس هل السؤال نفسه (مش الأدلة) قريب من تخصص السكر/الغدد الصماء."""
    q_emb = embedder.encode(
        [query], convert_to_numpy=True, normalize_embeddings=True
    )[0]
    sim = float(np.dot(q_emb, domain_centroid))
    return sim >= IN_DOMAIN_THRESHOLD

def passes_gate(query: str, reranked: pd.DataFrame) -> bool:
    """يرجّع True لو السؤال داخل التخصص والأدلة كافية، وFalse غير كده."""
    if not is_in_domain(query):
        return False

    if len(reranked) == 0:
        return False

    top = reranked.iloc[0]

    if float(top["rerank_score"]) < MIN_RERANK_SCORE:
        return False

    if float(top["similarity"]) < MIN_SIMILARITY_FLOOR:
        return False

    support = int((reranked["rerank_score"] >= SUPPORT_SCORE).sum())

    if support < MIN_SUPPORT_COUNT:
        return False

    return True


def medical_rag_answer(
    query: str,
    user_data: str = "",
    verbose: bool = True
) -> dict:

    candidates = vector_search(query, top_n=RETRIEVE_TOP_N)

    reranked = rerank(
        query,
        candidates,
        top_k=RERANK_TOP_K
    )

    top_score = (
        float(reranked["rerank_score"].max())
        if len(reranked)
        else -999
    )

    if passes_gate(query, reranked):   # <-- اتغيرت هنا: بقى بياخد query
        evidence = build_evidence(reranked)
    else:
        evidence = []

    answer = generate_answer(
        query,
        user_data,
        evidence
    )

    result = {
        "query": query,
        "user_data": user_data,
        "answer": answer,
        "evidence": evidence,
        "top_rerank_score": top_score,
    }

    if verbose:
        print("Q:", query)

        print("\n--- User Data ---")
        print(user_data if user_data else "(none)")

        print("\n--- Evidence used ---")

        if not evidence:
            print("(none — did not pass the confidence gate)")

        for e in evidence:
            print(
                f"[{e['doc_id']}] "
                f"sim={e['similarity']} "
                f"rerank={e['rerank_score']} | "
                f"{e['question']}"
            )

        print("\n--- Final Answer ---")
        print(textwrap.fill(answer, width=100))

    return result

In [100]:
_ = medical_rag_answer("What are the common symptoms of hypothyroidism?")

Q: What are the common symptoms of hypothyroidism?

--- User Data ---
(none)

--- Evidence used ---
[10924] sim=0.99 rerank=9.309 | What are the symptoms of hypothyroidism?
[12883] sim=0.99 rerank=8.24 | What are the symptoms of hypothyroidism?
[12438] sim=0.981 rerank=8.693 | Hello
What are the most prominent symptoms of hypothyroidism?
[15321] sim=0.957 rerank=7.477 | I want to know the symptoms of hypothyroidism
[14391] sim=0.949 rerank=7.562 | Hello, my question is: What are the symptoms that indicate hypothyroidism?
[14226] sim=0.937 rerank=7.942 | What are the symptoms of hypothyroidism

--- Final Answer ---
Common symptoms of hypothyroidism include:  - Extreme fatigue or general laziness and
lethargy [10924][12438][15321][14391][14226]   - Sensitivity to cold / feeling
cold [10924][12883][12438][14391][14226]   - Weight gain [10924][12883][12438][15321][14391][14226]
- Constipation [10924][12438]   - Depression or mood changes [10924]   - Muscle pain, weakness,
spasms, or tensio

In [101]:
_ = medical_rag_answer("How much insulin should I take?")

Q: How much insulin should I take?

--- User Data ---
(none)

--- Evidence used ---
[2890] sim=0.767 rerank=7.13 | The amount of insulin I take is as follows: 25 units in the morning, 35 units in the afternoon, 25 units at night, and the daily blood sugar average is 300. Can these doses be increased, and do I need more than that?
[4794] sim=0.773 rerank=6.172 | I don't know why insulin doses are appropriate for me or not. I take 70 units a day, 40 units in the morning and 30 units in the evening.
[541] sim=0.726 rerank=5.837 | How much insulin should be in the blood? What is insulin A?
[2622] sim=0.726 rerank=3.524 | How much insulin should a diabetic patient inject if the sugar level rises above 300 ml?
[9334] sim=0.712 rerank=4.169 | I use 10 units of insulin 70/30 in the morning, but surprisingly, it does not exceed 99 or less, and after four or three hours, 45, and before breakfast at most, 89, I stopped eating all...
[3848] sim=0.742 rerank=1.425 | What is the recommended dose of 

In [102]:
_ = medical_rag_answer("What are the best treatments for a broken wrist?")

Q: What are the best treatments for a broken wrist?

--- User Data ---
(none)

--- Evidence used ---
(none — did not pass the confidence gate)

--- Final Answer ---
The available medical evidence does not provide enough information to answer this reliably.


In [103]:
_ = medical_rag_answer("I have knee pain when I walk, what could this be?")

Q: I have knee pain when I walk, what could this be?

--- User Data ---
(none)

--- Evidence used ---
(none — did not pass the confidence gate)

--- Final Answer ---
The available medical evidence does not provide enough information to answer this reliably.


In [104]:
_ = medical_rag_answer("Why do I feel short of breath and tired when I climb stairs?")

Q: Why do I feel short of breath and tired when I climb stairs?

--- User Data ---
(none)

--- Evidence used ---
(none — did not pass the confidence gate)

--- Final Answer ---
The available medical evidence does not provide enough information to answer this reliably.


# **Evaluation**

In [105]:
sample = df.sample(n=50, random_state=42)
self_hit = 0

for _, row in sample.iterrows():
    top1 = vector_search(row["Question"], top_n=1)
    if top1.iloc[0]["doc_id"] == row["doc_id"]:
        self_hit += 1

print(f"Self-retrieval accuracy (Top-1): {self_hit}/{len(sample)} = {self_hit/len(sample):.1%}")

Self-retrieval accuracy (Top-1): 50/50 = 100.0%


In [106]:
import numpy as np

question_groups = df.groupby("Question")["doc_id"].apply(set)
duplicate_questions = question_groups[question_groups.apply(len) >= 2]

print(f"Questions with duplicates usable for evaluation: {len(duplicate_questions)}")

def group_has_consistent_answers(doc_ids, threshold: float = 0.5) -> bool:
    answers = df[df["doc_id"].isin(doc_ids)]["Answer"].astype(str).tolist()
    if len(answers) < 2:
        return False
    ref = answers[0]
    return all(is_near_duplicate(ref, a, threshold=threshold) for a in answers[1:])

consistent_groups = duplicate_questions[
    duplicate_questions.apply(lambda ids: group_has_consistent_answers(ids))
]

print(f"Groups with consistent (answer-aligned) duplicates: {len(consistent_groups)}")

eligible_doc_ids = [
    doc_id
    for group in consistent_groups
    for doc_id in group
]

def evaluate_retrieval(
    top_k_values=(1, 3, 5, 10),
    n_queries=150,
    seed=42,
    use_reranker=False
):
    rng = np.random.RandomState(seed)
    n = min(n_queries, len(eligible_doc_ids))
    sample_ids = rng.choice(
        eligible_doc_ids,
        size=n,
        replace=False
    )

    max_k = max(top_k_values)

    recalls = {k: [] for k in top_k_values}
    precisions = {k: [] for k in top_k_values}
    f1_scores = {k: [] for k in top_k_values}

    reciprocal_ranks = []

    for doc_id in sample_ids:
        row = df[df["doc_id"] == doc_id].iloc[0]
        query_text = row["Question"]

        relevant_ids = consistent_groups[query_text] - {doc_id}

        if not relevant_ids:
            continue

        candidates = vector_search(
            query_text,
            top_n=max_k * 3
        )

        candidates = candidates[
            candidates["doc_id"] != doc_id
        ]

        if use_reranker:
            candidates = rerank(
                query_text,
                candidates,
                top_k=max_k
            )

        retrieved_ids = candidates["doc_id"].tolist()[:max_k]

        total_relevant = len(relevant_ids)

        for k in top_k_values:
            retrieved_at_k = retrieved_ids[:k]

            relevant_retrieved = sum(
                rid in relevant_ids
                for rid in retrieved_at_k
            )

            recall = (
                relevant_retrieved / total_relevant
                if total_relevant > 0
                else 0.0
            )

            precision = (
                relevant_retrieved / k
                if k > 0
                else 0.0
            )

            if recall + precision > 0:
                f1 = (
                    2 * precision * recall
                    / (precision + recall)
                )
            else:
                f1 = 0.0

            recalls[k].append(recall)
            precisions[k].append(precision)
            f1_scores[k].append(f1)

        rr = 0.0

        for rank, rid in enumerate(retrieved_ids, start=1):
            if rid in relevant_ids:
                rr = 1.0 / rank
                break

        reciprocal_ranks.append(rr)

    results = {}

    for k in top_k_values:
        results[f"Recall@{k}"] = round(
            np.mean(recalls[k]),
            3
        )

        results[f"Precision@{k}"] = round(
            np.mean(precisions[k]),
            3
        )

        results[f"F1@{k}"] = round(
            np.mean(f1_scores[k]),
            3
        )

    results["MRR"] = round(
        np.mean(reciprocal_ranks),
        3
    )

    results["n_queries"] = len(reciprocal_ranks)

    return results


print("Retrieval only:")
print(
    evaluate_retrieval(
        use_reranker=False
    )
)

print("\nRetrieval + Cross-Encoder Reranker:")
print(
    evaluate_retrieval(
        use_reranker=True
    )
)

Questions with duplicates usable for evaluation: 67
Groups with consistent (answer-aligned) duplicates: 13
Retrieval only:
{'Recall@1': np.float64(1.0), 'Precision@1': np.float64(1.0), 'F1@1': np.float64(1.0), 'Recall@3': np.float64(1.0), 'Precision@3': np.float64(0.333), 'F1@3': np.float64(0.5), 'Recall@5': np.float64(1.0), 'Precision@5': np.float64(0.2), 'F1@5': np.float64(0.333), 'Recall@10': np.float64(1.0), 'Precision@10': np.float64(0.1), 'F1@10': np.float64(0.182), 'MRR': np.float64(1.0), 'n_queries': 26}

Retrieval + Cross-Encoder Reranker:
{'Recall@1': np.float64(0.769), 'Precision@1': np.float64(0.769), 'F1@1': np.float64(0.769), 'Recall@3': np.float64(1.0), 'Precision@3': np.float64(0.333), 'F1@3': np.float64(0.5), 'Recall@5': np.float64(1.0), 'Precision@5': np.float64(0.2), 'F1@5': np.float64(0.333), 'Recall@10': np.float64(1.0), 'Precision@10': np.float64(0.1), 'F1@10': np.float64(0.182), 'MRR': np.float64(0.865), 'n_queries': 26}


In [107]:
SEMANTIC_SIM_THRESHOLD = 0.90  # عتبة عالية عمدًا عشان نقلل False Positives

def build_semi_automatic_gold_set(n_questions: int = 100, seed: int = 42) -> dict:
    """
    بيرجع dict:
        query -> {
            "relevant_ids": set(...),   # exact duplicates -> نثق فيها 100%
            "candidate_ids": set(...),  # قريبة معنويًا (مش نفس النص) -> تحتاج مراجعة يدوية
        }
    """
    rng = np.random.RandomState(seed)
    unique_questions = df["Question"].drop_duplicates().tolist()
    sample_questions = rng.choice(
        unique_questions,
        size=min(n_questions, len(unique_questions)),
        replace=False,
    )

    gold = {}
    for q in sample_questions:
        own_ids = set(df[df["Question"] == q]["doc_id"])

        candidates = vector_search(q, top_n=15)
        semantic_candidates = candidates[
            (candidates["similarity"] >= SEMANTIC_SIM_THRESHOLD)
            & (~candidates["doc_id"].isin(own_ids))
        ]

        gold[q] = {
            "relevant_ids": own_ids,
            "candidate_ids": set(semantic_candidates["doc_id"]),
        }
    return gold


gold_set = build_semi_automatic_gold_set(n_questions=100)

n_with_candidates = sum(1 for v in gold_set.values() if v["candidate_ids"])
print(f"Built a starter gold set for {len(gold_set)} questions.")
print(f"{n_with_candidates} of them have extra semantic candidates that need manual review.")

review_rows = []
for q, v in gold_set.items():
    for cid in v["candidate_ids"]:
        cand_row = df[df["doc_id"] == cid].iloc[0]
        review_rows.append({
            "query": q,
            "candidate_doc_id": cid,
            "candidate_question": cand_row["Question"],
            "candidate_answer": str(cand_row["Answer"])[:200],
            "is_relevant (fill manually: 1/0)": "",
        })

review_df = pd.DataFrame(review_rows)
review_path = "/content/gold_set_for_manual_review.xlsx"
review_df.to_excel(review_path, index=False)
print(f"Saved {len(review_df)} candidate pairs to {review_path} for manual labeling.")

Built a starter gold set for 100 questions.
3 of them have extra semantic candidates that need manual review.
Saved 16 candidate pairs to /content/gold_set_for_manual_review.xlsx for manual labeling.


In [108]:
def recall_at_k(retrieved_ids, relevant_ids, k):
    retrieved = retrieved_ids[:k]
    return int(len(set(retrieved) & set(relevant_ids)) > 0)


def reciprocal_rank(retrieved_ids, relevant_ids):
    relevant_ids = set(relevant_ids)
    for rank, doc_id in enumerate(retrieved_ids, start=1):
        if doc_id in relevant_ids:
            return 1.0 / rank
    return 0.0


def ndcg_at_k(retrieved_ids, relevant_ids, k):
    relevant_ids = set(relevant_ids)
    dcg = 0.0
    for i, doc_id in enumerate(retrieved_ids[:k], start=1):
        if doc_id in relevant_ids:
            dcg += 1.0 / np.log2(i + 1)
    ideal_hits = min(len(relevant_ids), k)
    idcg = sum(1.0 / np.log2(i + 1) for i in range(1, ideal_hits + 1))
    return dcg / idcg if idcg > 0 else 0.0


def evaluate_gold_retrieval(gold_set: dict, relevant_ids_map: dict, top_k: int = 5, retrieval_n: int = 30, use_reranker: bool = False):
    """
    gold_set:          الـ dict الأصلي (query -> {"relevant_ids": own_ids, ...}) --
                        بنستخدمه بس عشان نعرف نستبعد الصف/النسخ المطابقة حرفيًا من البحث.
    relevant_ids_map:   query -> مجموعة الصح الكاملة (الصف نفسه + الـ candidates المعتمدة بـ 1) --
                        بنستخدمه في حساب الـ score.
    """
    recalls, mrrs, ndcgs = [], [], []
    skipped_no_new_evidence = 0

    for query, relevant_ids in relevant_ids_map.items():
        own_ids = gold_set[query]["relevant_ids"]
        extra_relevant = relevant_ids - own_ids

        if not extra_relevant:
            skipped_no_new_evidence += 1
            continue

        candidates = vector_search(query, top_n=retrieval_n)
        candidates = candidates[~candidates["doc_id"].isin(own_ids)]  # استبعد الصف نفسه بس
        if use_reranker:
            candidates = rerank(query, candidates, top_k=top_k)
        else:
            candidates = candidates.head(top_k)
        retrieved_ids = candidates["doc_id"].tolist()[:top_k]

        recalls.append(recall_at_k(retrieved_ids, extra_relevant, top_k))
        mrrs.append(reciprocal_rank(retrieved_ids, extra_relevant))
        ndcgs.append(ndcg_at_k(retrieved_ids, extra_relevant, top_k))

    print(f"(تم تخطي {skipped_no_new_evidence} سؤال مفيهمش candidates معتمدة)")

    if not recalls:
        return {"note": "مفيش ولا سؤال عنده candidate معتمد بـ 1 -- راجع ملف الإكسل وتأكد إنك حطيت 1 في صفوف فعلاً relevant"}

    return {
        f"Recall@{top_k}": round(float(np.mean(recalls)), 3),
        "MRR": round(float(np.mean(mrrs)), 3),
        f"NDCG@{top_k}": round(float(np.mean(ndcgs)), 3),
        "n_queries": len(recalls),
    }


def merge_reviewed_gold_set(gold_set: dict, reviewed_path: str = "/content/gold_set_for_manual_review.xlsx") -> dict:
    """بعد ما تراجع ملف gold_set_for_manual_review.xlsx وتحط 1/0، شغّل الدالة دي
    عشان تضيف الـ candidates اللي عليهم 1 لمجموعة الـ relevant_ids النهائية.

    لو الملف لسه مترفعش (مثلاً وقت Run All كامل من غير توقف للمراجعة اليدوية)،
    بترجع الـ gold set الأساسي زي ما هو (exact duplicates بس) بدل ما تكسر الـ notebook
    بـ FileNotFoundError مش واضحة.
    """
    try:
        reviewed = pd.read_excel(reviewed_path)
    except FileNotFoundError:
        print(f"⚠️ {reviewed_path} مش موجود لسه. راجع الملف اللي اتصدّر فوق وحطه هنا الأول.")
        print("هرجع الـ gold set الأساسي (exact duplicates فقط) لحد ما تعمل المراجعة.")
        return {q: set(v["relevant_ids"]) for q, v in gold_set.items()}

    merged = {q: set(v["relevant_ids"]) for q, v in gold_set.items()}
    for _, r in reviewed.iterrows():
        if str(r["is_relevant (fill manually: 1/0)"]).strip() == "1":
            merged.setdefault(r["query"], set()).add(r["candidate_doc_id"])
    return merged


gold_evidence_final = merge_reviewed_gold_set(gold_set)

n_with_approved = sum(1 for q, ids in gold_evidence_final.items() if ids - gold_set[q]["relevant_ids"])
print(f"Total questions in gold set: {len(gold_evidence_final)}")
print(f"Questions with at least one manually-approved candidate: {n_with_approved}")

if n_with_approved == 0:
    print("\n⚠️ لسه معملتش تعديل في ملف gold_set_for_manual_review.xlsx (أو رفعته من غير حفظ).")
    print("افتح الملف، حط 1 في الصفوف اللي فعلاً relevant، ارفعه تاني على /content بنفس الاسم، وشغّل الخلية دي تاني.")
else:
    print("\nFinal Gold Retrieval only (bi-encoder / FAISS):")
    print(evaluate_gold_retrieval(gold_set, gold_evidence_final, top_k=5, use_reranker=False))

    print("\nFinal Gold Retrieval + Cross-Encoder Reranker:")
    print(evaluate_gold_retrieval(gold_set, gold_evidence_final, top_k=5, use_reranker=True))


Total questions in gold set: 100
Questions with at least one manually-approved candidate: 0

⚠️ لسه معملتش تعديل في ملف gold_set_for_manual_review.xlsx (أو رفعته من غير حفظ).
افتح الملف، حط 1 في الصفوف اللي فعلاً relevant، ارفعه تاني على /content بنفس الاسم، وشغّل الخلية دي تاني.


In [109]:
def category_match_recall(top_k: int = 5, n_queries: int = 100, seed: int = 42, use_reranker: bool = True):
    sample = df.sample(n=min(n_queries, len(df)), random_state=seed)
    hits = []

    for _, row in sample.iterrows():
        query = row["Question"]
        true_category = row["Category"]

        candidates = vector_search(query, top_n=20)
        candidates = candidates[candidates["doc_id"] != row["doc_id"]]  # leave-one-out بالـ id بدل نص الـ Question

        if use_reranker:
            candidates = rerank(query, candidates, top_k=top_k)
        else:
            candidates = candidates.head(top_k)

        hit = candidates["Category"].astype(str).eq(str(true_category)).any()
        hits.append(hit)

    return {"category_recall": round(float(np.mean(hits)), 3), "n": len(hits)}


print("Category sanity check (retriever + reranker):")
print(category_match_recall(use_reranker=True))
print("\nCategory sanity check (retriever only):")
print(category_match_recall(use_reranker=False))

Category sanity check (retriever + reranker):
{'category_recall': 0.97, 'n': 100}

Category sanity check (retriever only):
{'category_recall': 0.97, 'n': 100}


In [116]:
eval_set = [
    #Positive
    ("What are the symptoms of diabetes?", True),
    ("What is the treatment for diabetes?", True),
    ("Is there a cure for diabetes?", True),
    ("What is the normal blood sugar level?", True),
    ("What are the causes of diabetes?", True),
    ("What foods are beneficial for diabetics?", True),
    ("What is diabetic foot?", True),
    ("What are the symptoms of thyroid disease?", True),
    ("What is the treatment for hypothyroidism?", True),
    ("I suffer from an underactive thyroid gland", True),
    ("What are the symptoms of high blood sugar?", True),
    ("What is the treatment for low blood sugar?", True),
    ("What are the symptoms of gestational diabetes?", True),
    ("What are the risk factors for type 1 diabetes?", True),
    ("What is the treatment for hyperthyroidism?", True),
    ("Can diabetes be cured permanently?", True),
    ("What is a normal HbA1c level?", True),
    ("What are the complications of untreated diabetes?", True),
    ("Is it normal for TSH to be slightly high after thyroid surgery?", True),
    ("What is the relationship between diabetes and weight loss?", True),
    ("How is hypothyroidism diagnosed?", True),
    ("What are the symptoms of hyperthyroidism?", True),
    ("What is the treatment for diabetic foot?", True),
    ("What causes weight gain in hypothyroidism?", True),
    ("Is thyroid disease hereditary?", True),

    #Negative
    ("What is the best treatment for a broken leg?", False),
    ("How do I fix a slow laptop?", False),
    ("What is the capital of France?", False),
    ("Best recipe for chocolate cake?", False),
    ("How to change a car tire?", False),
    ("What is the treatment for a migraine headache?", False),
    ("How do I treat a common cold?", False),
    ("What are the symptoms of appendicitis?", False),
    ("How to remove a stain from a carpet?", False),
    ("What is the best programming language to learn?", False),
    ("How do I train for a marathon?", False),
    ("What is the treatment for a skin rash from poison ivy?", False),
    ("How do I apply for a passport?", False),
    ("What are the symptoms of a kidney stone?", False),
    ("Best way to whiten teeth at home?", False),
    ("How do I treat a sprained ankle?", False),
    ("What is the weather like today?", False),
    ("How to fix a leaking faucet?", False),
    ("What is the treatment for asthma?", False),
    ("How do I improve my credit score?", False),
    ("What is the treatment for a broken arm?", False),
    ("How do I treat a sunburn?", False),
    ("What's the best laptop for gaming?", False),
    ("How do I remove a splinter?", False),
    ("What is the treatment for food poisoning?", False),
]

rows = []
for query, should_answer in eval_set:
    candidates = vector_search(query, top_n=RETRIEVE_TOP_N)
    reranked = rerank(query, candidates, top_k=RERANK_TOP_K)
    top_score = float(reranked["rerank_score"].max()) if len(reranked) else -999
    system_answered = passes_gate(query, reranked)
    rows.append({
        "query": query,
        "expected_answer": should_answer,
        "system_answered": system_answered,
        "top_rerank_score": round(top_score, 3),
        "correct_decision": should_answer == system_answered,
    })

gate_df = pd.DataFrame(rows)
print(gate_df.to_string(index=False))
print(f"\nGate decision accuracy: {gate_df['correct_decision'].mean():.1%}")

print("\ntop_rerank_score by expected label:")
print(gate_df.groupby("expected_answer")["top_rerank_score"].describe()[["mean", "min", "max"]])

                                                          query  expected_answer  system_answered  top_rerank_score  correct_decision
                             What are the symptoms of diabetes?             True             True             9.837              True
                            What is the treatment for diabetes?             True             True             9.100              True
                                  Is there a cure for diabetes?             True             True             9.930              True
                          What is the normal blood sugar level?             True             True             8.873              True
                               What are the causes of diabetes?             True             True             9.455              True
                       What foods are beneficial for diabetics?             True             True             9.858              True
                                         What is diabetic foot

In [117]:
def find_best_threshold_cv(scores, labels, n_folds: int = 5, n_steps: int = 200, seed: int = 42):
    scores = np.array(scores, dtype=float)
    labels = np.array(labels, dtype=int)
    n = len(scores)
    n_folds = min(n_folds, n)

    rng = np.random.RandomState(seed)
    idx = rng.permutation(n)
    folds = np.array_split(idx, n_folds)

    fold_thresholds, fold_test_accuracies = [], []

    for i in range(n_folds):
        test_idx = folds[i]
        train_idx = np.concatenate([folds[j] for j in range(n_folds) if j != i])
        if len(train_idx) == 0 or len(test_idx) == 0:
            continue

        train_scores, train_labels = scores[train_idx], labels[train_idx]
        test_scores, test_labels = scores[test_idx], labels[test_idx]

        best_threshold, best_acc = None, -1.0
        for t in np.linspace(scores.min(), scores.max(), n_steps):
            preds = (train_scores >= t).astype(int)
            acc = float(np.mean(preds == train_labels))
            if acc > best_acc:
                best_acc, best_threshold = acc, float(t)

        test_preds = (test_scores >= best_threshold).astype(int)
        test_acc = float(np.mean(test_preds == test_labels))

        fold_thresholds.append(best_threshold)
        fold_test_accuracies.append(test_acc)

    return {
        "fold_thresholds": [round(t, 3) for t in fold_thresholds],
        "mean_threshold": round(float(np.mean(fold_thresholds)), 3),
        "std_threshold": round(float(np.std(fold_thresholds)), 3),
        "fold_test_accuracies": [round(a, 3) for a in fold_test_accuracies],
        "mean_test_accuracy": round(float(np.mean(fold_test_accuracies)), 3),
        "std_test_accuracy": round(float(np.std(fold_test_accuracies)), 3),
    }


cv_result = find_best_threshold_cv(
    scores=gate_df["top_rerank_score"].tolist(),
    labels=gate_df["expected_answer"].astype(int).tolist(),
    n_folds=5,
)
print("Cross-validated threshold tuning:")
for k, v in cv_result.items():
    print(f"  {k}: {v}")

print(
    f"\nمتوسط الـ threshold المقترح: {cv_result['mean_threshold']} "
    f"(± {cv_result['std_threshold']}), بمتوسط accuracy على test folds = "
    f"{cv_result['mean_test_accuracy']:.1%} (± {cv_result['std_test_accuracy']:.1%})"
)
print(
    "لو الـ std كبير (يعني القيمة بتتغير كتير بين fold وfold)، ده معناه العينة "
    "لسه صغيرة ومحتاجة توسّع أكتر قبل ما تثق في threshold نهائي."
)


Cross-validated threshold tuning:
  fold_thresholds: [1.147, 1.147, -0.875, 1.147, 1.147]
  mean_threshold: 0.742
  std_threshold: 0.809
  fold_test_accuracies: [0.9, 1.0, 0.9, 1.0, 1.0]
  mean_test_accuracy: 0.96
  std_test_accuracy: 0.049

متوسط الـ threshold المقترح: 0.742 (± 0.809), بمتوسط accuracy على test folds = 96.0% (± 4.9%)
لو الـ std كبير (يعني القيمة بتتغير كتير بين fold وfold)، ده معناه العينة لسه صغيرة ومحتاجة توسّع أكتر قبل ما تثق في threshold نهائي.


In [118]:
import time

def medical_rag_answer_timed(query: str) -> dict:
    t0 = time.perf_counter()
    candidates = vector_search(query, top_n=RETRIEVE_TOP_N)
    t1 = time.perf_counter()

    reranked = rerank(query, candidates, top_k=RERANK_TOP_K)
    t2 = time.perf_counter()

    top_score = float(reranked["rerank_score"].max()) if len(reranked) else -999
    evidence = build_evidence(reranked) if passes_gate(query, reranked) else []  # <-- اتغيرت هنا
    t3 = time.perf_counter()

    answer = generate_answer(query, "", evidence)
    t4 = time.perf_counter()

    return {
        "query": query, "answer": answer, "evidence": evidence,
        "timing_sec": {
            "retrieval": round(t1 - t0, 3),
            "rerank": round(t2 - t1, 3),
            "evidence_build": round(t3 - t2, 3),
            "llm_generation": round(t4 - t3, 3),
            "total": round(t4 - t0, 3),
        },
    }


result = medical_rag_answer_timed("What are the symptoms of diabetes?")
print(result["timing_sec"])

{'retrieval': 0.015, 'rerank': 0.097, 'evidence_build': 0.02, 'llm_generation': 1.432, 'total': 1.565}


# **SAVE**

In [119]:
import os
import json
import shutil

SAVE_DIR = "/content/rag_model"

os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(f"{SAVE_DIR}/faiss", exist_ok=True)
os.makedirs(f"{SAVE_DIR}/models", exist_ok=True)
os.makedirs(f"{SAVE_DIR}/data", exist_ok=True)

faiss.write_index(
    faiss_index,
    f"{SAVE_DIR}/faiss/questions.index"
)

print("✓ FAISS index saved")


# 2) Save embeddings
np.save(
    f"{SAVE_DIR}/faiss/question_embeddings.npy",
    question_embeddings
)

print("✓ Embeddings saved")


# 3) Save dataset
df.to_pickle(
    f"{SAVE_DIR}/data/knowledge_base.pkl"
)

print("✓ Knowledge base saved")


# 4) Save embedding model name
with open(
    f"{SAVE_DIR}/models/embedding_model.txt",
    "w"
) as f:
    f.write(EMBEDDING_MODEL_NAME)

print("✓ Embedding model config saved")


# 5) Save reranker model name
with open(
    f"{SAVE_DIR}/models/reranker_model.txt",
    "w"
) as f:
    f.write(RERANKER_MODEL_NAME)

print("✓ Reranker model config saved")


# 6) Save medical synonyms
with open(
    f"{SAVE_DIR}/models/medical_synonyms.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        MEDICAL_SYNONYMS,
        f,
        ensure_ascii=False,
        indent=2
    )

print("✓ Medical synonyms saved")

# 7) Save RAG configuration
rag_config = {
    "embedding_model": EMBEDDING_MODEL_NAME,
    "reranker_model": RERANKER_MODEL_NAME,

    "retrieve_top_n": RETRIEVE_TOP_N,
    "rerank_top_k": RERANK_TOP_K,

    "min_rerank_score": MIN_RERANK_SCORE,
    "min_similarity_floor": MIN_SIMILARITY_FLOOR,
    "min_support_count": MIN_SUPPORT_COUNT,
    "support_score": SUPPORT_SCORE,

    "semantic_similarity_threshold": SEMANTIC_SIM_THRESHOLD,

    "embedding_dimension": int(embedding_dim),

    "dataset_size": int(len(df))
}

with open(
    f"{SAVE_DIR}/rag_config.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        rag_config,
        f,
        indent=2
    )

print("✓ RAG configuration saved")


# 8) Save requirements
requirements = """sentence-transformers
faiss-cpu
pandas
numpy
openpyxl
groq
tqdm
"""

with open(
    f"{SAVE_DIR}/requirements.txt",
    "w"
) as f:
    f.write(requirements)

print("✓ Requirements saved")


# 9) Summary
print("RAG PIPELINE SAVED SUCCESSFULLY")

for root, dirs, files in os.walk(SAVE_DIR):
    level = root.replace(SAVE_DIR, "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")

    for file in files:
        print(f"{indent}  └── {file}")

✓ FAISS index saved
✓ Embeddings saved
✓ Knowledge base saved
✓ Embedding model config saved
✓ Reranker model config saved
✓ Medical synonyms saved
✓ RAG configuration saved
✓ Requirements saved
RAG PIPELINE SAVED SUCCESSFULLY
rag_model/
  └── requirements.txt
  └── rag_config.json
  models/
    └── embedding_model.txt
    └── medical_synonyms.json
    └── reranker_model.txt
  faiss/
    └── question_embeddings.npy
    └── questions.index
  data/
    └── knowledge_base.pkl
    └── metadata_store.json


In [120]:
# Save metadata_store
with open(f"{SAVE_DIR}/data/metadata_store.json", "w", encoding="utf-8") as f:
    json.dump(metadata_store, f, ensure_ascii=False, indent=2)

print("✓ Metadata store saved")

✓ Metadata store saved


In [121]:
from google.colab import files

files.download("/content/questions.index")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>